Bei anderen NB doppelte resample ünberprüfen dank prune

## Imports

In [57]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import model_analysis

import importlib
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

import numpy as np

import cartopy.crs as ccrs
from scipy.stats import beta



## Load Data

In [58]:
max_radius = 4
hist = 1
is_local_data = False
Month_idx = 4
safe = True
start_wanted = None  # later this will be shifted, if it is to close to the beginning of the data, such that there is allways data also for the hist dimension
end = None
max_depth = 3


In [59]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [60]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [61]:

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
approx_maximas = max_v


In [62]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=3)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=3)

raw_input_for_skew = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

 

In [63]:
#min_start 

data_start = pd.to_datetime(raw_mrsol_for_mean.time[0].item())
offset = pd.tseries.frequencies.to_offset("ME")
min_start = data_start + pd.DateOffset(months=hist)

if start_wanted is None:
    start = min_start
else:
    start = pd.to_datetime(start_wanted) 
    start = max(min_start,start)
start = start.strftime("%Y-%m-%d")  

some explantions
slice 0-4 weil im moment die letzte schicht hartnäckig probleme macht

In [64]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas).clip(max = 1-1e-15)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [65]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [66]:
# Transform already as one function
#model.transform.Logit_Transform_ds()

In [67]:
def shape_input(ds, chunk_mask, hist, radius):
    ds = model.shape_data.add_hist_dimension(ds, hist)    
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.shape_data.add_radius_dimension(ds, radius)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

In [68]:
#The chunk_mask schould all be the same, this is important that there are no shape problems in the regressions, you can test this with  
#(chunk_mask != chunk_mask_2).sum(),#(chunk_mask != chunk_mask_1).sum(),
#The chunk_mask is used, that the regression can predict a depth.size vector at each gridcell, the masks: detail_mask_mean, detail_mask_var and detail_mask_test are later used to determan which points are real predictions and witch where just placeholders to let the regression run smoothly.

### Linear Regression of the mean

In [69]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [70]:
mean_predictor_sets  = {}
mean_max_predictor = shape_input(raw_input_for_mean,chunk_mask,hist, max_radius)
mean_max_predictor
for radius in range(0, max_radius):
   mean_predictor_sets[f"radius_{radius}"] = mean_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [71]:
mean_predictor_sets["radius_2"]

<xarray.Dataset> Size: 194MB
Dimensions:           (hist: 1, time: 165, lon_translations: 5,
                       lat_translations: 5, gridcell: 2935)
Coordinates:
  * hist              (hist) int64 8B 0
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
  * lon_translations  (lon_translations) int64 40B -2 -1 0 1 2
  * lat_translations  (lat_translations) int64 40B -2 -1 0 1 2
    lat               (gridcell) float64 23kB -56.25 -56.25 ... 81.25 81.25
    lon               (gridcell) float64 23kB 288.8 291.2 293.8 ... 296.2 298.8
    height            float64 8B 2.0
Dimensions without coordinates: gridcell
Data variables:
    tas               (lat_translations, lon_translations, time, hist, gridcell) float64 97MB ...
    pr                (lat_translations, lon_translations, time, hist, gridcell) float64 97MB ...
Attributes: (12/52)
    CDI:                       Climate Data Interface version 1.9.6 (http://m...
    history:                   Thu Dec 19 16:54:57 2019: cdo -O -b F64 -remap...
    source:                    MPI-ESM1.2-LR (2017): \naerosol: none, prescri...
    institution:               Max Planck Institute for Meteorology
    Conventions:               CF-1.7 CMIP-6.2
    activity_id:               CMIP
    ...                        ...
    cmor_version:              3.5.0
    tracking_id:               hdl:21.14100/6b679cba-17b8-45eb-90dc-23d170c1998c
    cmip6-ng:                  \ncontact = cmip6-archive@env.ethz.ch\ndescrip...
    original_file_names:       /net/atmos/data/cmip6/historical/Amon/tas/MPI-...
    original_file_hash_codes:  44b9ee9e68daceb1f50e9680dcfb7744f0e272d73c5e18...
    CDO:                       Climate Data Operators version 1.9.6 (http://m...

In [72]:
Regr_set_mean = {}

In [73]:
#mean_target_da

In [74]:
for key,predictors in mean_predictor_sets.items():
    Regr_set_mean[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_mean[key].fit(predictors=predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

In [75]:
#Regr_set_mean["kontrolle"] = model.stats._parallel_linear_regression.ParLinearRegression()
#Regr_set_mean["kontrolle"].fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

Hier wäre die meinung einen neuen Run zu verwenden, die frage ist ob sich die variance durch das fehlen des runs zu tief ausfällt. Overfitting korrektur mit 1/(1-param/n_samples)^2? (SPäter probieren)

In [76]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)#Hier können noch sehr grosse werte auftauchen, wenn in irgendwelchen schichten die Maximas der verschieden runs sehr unterschiedlich sind.

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [77]:
var_predictor_sets  = {}
var_max_predictor = shape_input(raw_input_for_var,chunk_mask,hist,max_radius)

for radius in range(0, max_radius):
    var_predictor_sets[f"radius_{radius}"] = var_max_predictor.sel(lon_translations=slice(-radius,radius)).sel(lat_translations=slice(-radius,radius))

In [78]:
Regr_set_mean["radius_1"].params.pr

<xarray.DataArray 'pr' (gridcell: 2935, depth: 3, hist: 1, lon_translations: 3,
                        lat_translations: 3)> Size: 634kB
array([[[[[ 8.18935961e+03,  1.25423919e+04, -4.30071323e+03],
          [-3.21108724e+02,  3.97455719e+03, -1.13316916e+04],
          [-1.52949291e+03, -5.05669553e+03,  1.12022122e+04]]],


        [[[-1.21684368e+01,  1.25198410e+03, -6.35129618e+02],
          [ 2.56060986e+02, -5.49483689e+02,  4.81034942e+02],
          [-3.24300189e+02,  3.27012279e+01, -1.44996039e+02]]],


        [[[-4.06280279e+02,  4.11551457e+00, -8.91626385e+01],
          [ 1.05754100e+02, -4.41739980e+01,  1.41899402e+02],
          [ 1.40305661e+02, -2.57553869e+02,  1.63095233e+02]]]],



       [[[[ 1.02993043e+04, -7.65742912e+03, -1.66888203e+03],
          [ 1.11740131e+03,  2.18597308e+04, -6.34601138e+03],
          [-1.72862012e+03, -1.60313243e+03, -3.44081728e+02]]],

...
        [[[-3.60583318e+02, -1.47333442e+04,  2.27408315e+04],
          [ 4.41068582e+04,  4.95644678e+04, -6.32305793e+04],
          [-4.83683833e+04, -2.89853674e+04,  4.34216989e+04]]]],



       [[[[-3.39414150e+03,  2.28596931e+04, -3.11268875e+04],
          [ 2.49037293e+03, -3.19015572e+04,  4.44491112e+04],
          [-2.24859524e+03,  1.73834197e+04, -1.67480195e+04]]],


        [[[-4.72668888e+03,  2.96293682e+04, -3.91385905e+04],
          [ 3.39402516e+03, -4.00102700e+04,  5.62797505e+04],
          [-2.67816462e+03,  2.09700981e+04, -2.14461197e+04]]],


        [[[-1.63610917e+04,  8.35209645e+04, -1.00376489e+05],
          [ 1.12326307e+04, -1.02071825e+05,  1.47165804e+05],
          [-5.85744732e+03,  4.68203273e+04, -5.78170110e+04]]]]],
      shape=(2935, 3, 1, 3, 3))
Coordinates:
  * gridcell          (gridcell) int64 23kB 0 1 2 3 4 ... 2931 2932 2933 2934
  * depth             (depth) float64 24B 0.03 0.19 0.78
  * hist              (hist) int64 8B 0
  * lon_translations  (lon_translations) int64 24B -1 0 1
  * lat_translations  (lat_translations) int64 24B -1 0 1

In [79]:
var_predictor_sets["radius_1"].pr 

<xarray.DataArray 'pr' (lat_translations: 3, lon_translations: 3, time: 165,
                        hist: 1, gridcell: 2935)> Size: 35MB
array([[[[[3.74067268e-05, 3.15918571e-05, 2.59698147e-05, ...,
           2.57311266e-06, 2.64105903e-06, 2.76036707e-06]],

         [[3.33056642e-05, 3.46473575e-05, 3.18068610e-05, ...,
           1.45211669e-06, 1.51715410e-06, 1.80223759e-06]],

         [[4.02724130e-05, 3.24345159e-05, 2.38353785e-05, ...,
           3.03153650e-06, 2.72001014e-06, 2.65284682e-06]],

         ...,

         [[5.23613938e-05, 4.61104072e-05, 3.83939059e-05, ...,
           4.63268958e-06, 5.22318576e-06, 5.11863468e-06]],

         [[3.24691551e-05, 3.44360406e-05, 3.54456449e-05, ...,
           2.67585901e-06, 2.81395225e-06, 2.73975893e-06]],

         [[3.87042206e-05, 2.88555813e-05, 2.65062841e-05, ...,
           3.46886291e-06, 3.35346970e-06, 3.43880002e-06]]],

...
        [[[5.19838722e-05, 4.95493698e-05, 4.40645828e-05, ...,
           5.31586880e-06, 3.38204272e-06, 2.79200658e-06]],

         [[2.92786864e-05, 3.11856284e-05, 3.34468870e-05, ...,
           7.22993985e-06, 7.21362033e-06, 7.18696865e-06]],

         [[3.98740030e-05, 4.29296893e-05, 4.05336558e-05, ...,
           6.19620122e-06, 5.64208452e-06, 5.48864929e-06]],

         ...,

         [[3.37495044e-05, 3.43165026e-05, 3.44941967e-05, ...,
           3.85702779e-06, 3.81058004e-06, 3.71031421e-06]],

         [[2.85960709e-05, 3.09541503e-05, 3.04478196e-05, ...,
           2.41997594e-06, 2.10759224e-06, 2.22589840e-06]],

         [[3.61427111e-05, 3.73076212e-05, 3.53376506e-05, ...,
           1.93393787e-06, 1.85120958e-06, 1.97769925e-06]]]]],
      shape=(3, 3, 165, 1, 2935))
Coordinates:
  * lat_translations  (lat_translations) int64 24B -1 0 1
  * lon_translations  (lon_translations) int64 24B -1 0 1
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
  * hist              (hist) int64 8B 0
    lat               (gridcell) float64 23kB -56.25 -56.25 ... 81.25 81.25
    lon               (gridcell) float64 23kB 288.8 291.2 293.8 ... 296.2 298.8
    height            float64 8B 2.0
Dimensions without coordinates: gridcell
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [80]:
Regr_set_mean["radius_2"].params.pr * var_predictor_sets["radius_2"].pr 

<xarray.DataArray 'pr' (gridcell: 2935, depth: 3, hist: 1, lon_translations: 5,
                        lat_translations: 5, time: 165)> Size: 291MB
array([[[[[[ 4.29345399e-02,  6.93683796e-02,  8.95113714e-02, ...,
             1.52628034e-01,  1.43921187e-01,  7.93303769e-02],
           [ 9.94771174e-03,  1.09098343e-02,  1.02130499e-02, ...,
             1.45193439e-02,  1.08432943e-02,  9.08610735e-03],
           [-4.42187810e-01, -4.52643372e-01, -3.76216749e-01, ...,
            -4.89394189e-01, -3.85346258e-01, -3.58242819e-01],
           [ 7.80884479e-02,  7.20390920e-02,  7.18626850e-02, ...,
             6.42749163e-02,  5.67185261e-02,  6.23779440e-02],
           [-8.86618126e-02, -7.69128533e-02, -9.62313231e-02, ...,
            -6.06460309e-02, -5.78156133e-02, -4.62880609e-02]],

          [[-1.91670906e-01, -2.60678469e-01, -3.46148579e-01, ...,
            -6.39035642e-01, -4.18408797e-01, -2.59044727e-01],
           [ 4.83270518e-01,  4.30287464e-01,  5.20293315e-01, ...,
             6.76475064e-01,  4.19480311e-01,  5.00033292e-01],
           [ 8.81774531e-01,  8.29384854e-01,  7.57133981e-01, ...,
             8.82486101e-01,  6.51738728e-01,  7.74793860e-01],
           [-9.32013881e-02, -7.07438060e-02, -8.57330932e-02, ...,
            -7.29589801e-02, -6.44004520e-02, -7.47429769e-02],
           [-4.60860497e-01, -3.56343350e-01, -5.67181334e-01, ...,
...
           [ 1.16260897e-01,  6.56109593e-02,  1.36973853e-01, ...,
             2.09318720e-01,  1.20903284e-01,  1.56733563e-01],
           [ 1.86015726e-01,  2.06525394e-01,  2.23483110e-01, ...,
             3.82568582e-01,  1.12899254e-01,  2.07817978e-01],
           [-2.72021413e-01, -7.00216603e-01, -5.34751652e-01, ...,
            -3.61490879e-01, -2.16866261e-01, -1.92684554e-01],
           [-1.91468847e-02, -1.05560225e-01, -8.46232064e-02, ...,
            -1.71447976e-02, -3.06942003e-02, -2.97467303e-02]],

          [[ 2.92235594e-03,  6.55890857e-04,  1.84870804e-03, ...,
             2.61683207e-03,  2.40175222e-03,  2.36169104e-03],
           [-1.74694267e-01, -9.05928357e-02, -1.87239670e-01, ...,
            -2.70329910e-01, -1.69525461e-01, -2.32598429e-01],
           [-1.37081292e-01, -1.27839231e-01, -1.23558814e-01, ...,
            -1.85131255e-01, -6.16765007e-02, -1.17275754e-01],
           [ 1.20714124e-01,  2.57473347e-01,  2.01381043e-01, ...,
             1.36009764e-01,  7.52255877e-02,  6.60746069e-02],
           [ 2.66602128e-02,  1.39838854e-01,  1.55513955e-01, ...,
             2.95912884e-02,  4.79241281e-02,  4.74788185e-02]]]]]],
      shape=(2935, 3, 1, 5, 5, 165))
Coordinates:
  * gridcell          (gridcell) int64 23kB 0 1 2 3 4 ... 2931 2932 2933 2934
    lat               (gridcell) float64 23kB -56.25 -56.25 ... 81.25 81.25
    lon               (gridcell) float64 23kB 288.8 291.2 293.8 ... 296.2 298.8
  * depth             (depth) float64 24B 0.03 0.19 0.78
  * hist              (hist) int64 8B 0
  * lon_translations  (lon_translations) int64 40B -2 -1 0 1 2
  * lat_translations  (lat_translations) int64 40B -2 -1 0 1 2
  * time              (time) datetime64[ns] 1kB 1850-04-30 ... 2014-04-30
    height            float64 8B 2.0
Attributes:
    standard_name:  precipitation_flux
    long_name:      Precipitation
    units:          kg m-2 s-1
    comment:        includes both liquid and solid phases
    original_name:  pr
    cell_methods:   area: time: mean
    cell_measures:  area: areacella

In [81]:
residuals = {}
for key, regr in Regr_set_mean.items():
    residuals[key] = regr.residuals(var_predictor_sets[key], var_target,location_dim="gridcell", regr_dim="time")


### Linear Regression of the Variance

In [82]:
Regr_set_var = {}

In [83]:
for key, res in residuals.items():
    Regr_set_var[key] = model.stats._parallel_linear_regression_stack.ParLinearRegressionStack(predictor_dims=["hist","lon_translations","lat_translations"])
    Regr_set_var[key].fit(predictors=var_predictor_sets[key], target=(res.residuals)**2,location_dim="gridcell", regr_dim="time")

for simplicity not in use
### Compute skewness samples
skew_target_noneT = shape_target(raw_mrsol_for_skew, approx_maximas)
skew_target_noneT, chunk_mask_skew, detail_mask_skew = mask_stack_target(skew_target_noneT)
skew_target = model.transform.Logit_Transform_ds(skew_target_noneT)

skew_predictors = shape_input(raw_input_for_skew,chunk_mask)
mean_prediction = LinReg_mean.predict(skew_predictors)
residuals = skew_target.mrsol - mean_prediction.prediction
sigmas = np.sqrt(LinReg_variance.predict(skew_predictors).prediction.clip(min = 1e-32))
standardized_values_for_skew = (residuals/sigmas)
### Linear Regression of the Skewness
LinReg_skewness = model.stats._parallel_linear_regression.ParLinearRegression()
LinReg_skewness.fit(predictors=skew_predictors, target=(standardized_values_for_skew)**3,location_dim="gridcell", regr_dim="time")

### Predictor

### Export Prameters

In [84]:
if safe:
    for key, mean_regr in Regr_set_mean.items():
        model.save.save_params(mean_regr.params,Regr_set_var[key].params, maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/radius/{key}/local{is_local_data}/month{Month_idx}", name=f"hist={hist},start={start_wanted},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
